<a href="https://colab.research.google.com/github/paulheather147/FinalYearProject/blob/main/FinalModel2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [46]:
!pip install -q datasets

import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.applications import VGG16, EfficientNetB2
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_preprocess
from datasets import load_dataset

import numpy as np


In [47]:
dataset = load_dataset("Falah/Alzheimer_MRI")
print(dataset)
print(dataset["train"].features)

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 5120
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 1280
    })
})
{'image': Image(mode=None, decode=True), 'label': ClassLabel(names=['Mild_Demented', 'Moderate_Demented', 'Non_Demented', 'Very_Mild_Demented'])}


In [48]:
train_val_split = dataset["train"].train_test_split(test_size=0.2, seed=42)
train = train_val_split["train"]
val = train_val_split["test"]
test = dataset["test"]
#smaller datasets to speed up training for debugging purposes
small_train = train.select(range(800))
small_val = val.select(range(200))
small_test = test.select(range(200))


def ensure_channel_dim(images):
    current_rank = tf.rank(images)

    def add_channel():
        return tf.expand_dims(images, axis=-1)

    def keep_same():
        return images

    return tf.cond(tf.equal(current_rank, 2), add_channel, keep_same)


def ensure_rgb_channels(images):
    current_channels = tf.shape(images)[-1]

    def to_rgb():
        return tf.image.grayscale_to_rgb(images)

    def drop_alpha():
        return images[..., :3]

    def keep_same():
        return images

    images = tf.cond(tf.equal(current_channels, 1), to_rgb, keep_same)

    new_channels = tf.shape(images)[-1]

    def keep_images_rgb():
        return images

    images = tf.cond(tf.equal(new_channels, 4), drop_alpha, keep_images_rgb)
    return images

def to_tensorflow_dataset(dataset_split, image_size, preprocess_fn, shuffle=False):
    dataset_tf = dataset_split.to_tf_dataset(
        columns=["image", "label"],
        shuffle=shuffle,
        num_workers=0,
    )

    def preprocess_input(example_dict):
        images = tf.cast(example_dict["image"], tf.float32)
        images = ensure_channel_dim(images)
        images = ensure_rgb_channels(images)
        images.set_shape([None, None, 3])
        images = tf.image.resize(images, (image_size, image_size))
        images = preprocess_fn(images)

        labels = tf.cast(example_dict["label"], tf.int32)
        return images, labels


    dataset_tf = dataset_tf.map(preprocess_input,
                                num_parallel_calls=tf.data.AUTOTUNE)
    dataset_tf = dataset_tf.batch(32)
    return dataset_tf.prefetch(tf.data.AUTOTUNE)

vgg_train_dataset = to_tensorflow_dataset(small_train, 224, vgg_preprocess, shuffle=True)
vgg_val_dataset = to_tensorflow_dataset(small_val,   224, vgg_preprocess, shuffle=False)
vgg_test_dataset = to_tensorflow_dataset(small_test,  224, vgg_preprocess, shuffle=False)

eff_train_dataset = to_tensorflow_dataset(small_train, 260, eff_preprocess, shuffle=True)
eff_val_dataset = to_tensorflow_dataset(small_val, 260, eff_preprocess, shuffle=False)
eff_test_dataset = to_tensorflow_dataset(small_test, 260, eff_preprocess, shuffle=False)


In [49]:
vgg_base = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3),
)

vgg_base.trainable = False

vgg_inputs = tf.keras.Input(shape=(224, 224, 3))
x = vgg_base(vgg_inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.5)(x)
vgg_outputs = layers.Dense(4, activation="softmax")(x)

vgg_model = tf.keras.Model(vgg_inputs, vgg_outputs)

vgg_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

In [50]:
VGG_EPOCHS = 1

vgg_history = vgg_model.fit(
    vgg_train_dataset,
    validation_data = vgg_val_dataset,
    epochs = VGG_EPOCHS,
)


25/25 ━━━━━━━━━━━━━━━━━━━━ 392s 16s/step - accuracy: 0.3988 - loss: 4.0575 - val_accuracy: 0.5450 - val_loss: 2.0943


In [51]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_true = []
y_pred = []

for images, labels in vgg_test_dataset:
    preds = vgg_model.predict(images)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred))

1/1 ━━━━━━━━━━━━━━━━━━━━ 12s 12s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 12s 12s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 12s 12s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 12s 12s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 12s 12s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 12s 12s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
[[ 0  0 32  1]
 [ 0  0  2  0]
 [ 0  0 85  5]
 [ 1  0 68  6]]
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        33
           1       0.00      0.00      0.00         2
           2       0.45      0.94      0.61        90
           3       0.50      0.08      0.14        75

    accuracy                           0.46       200
   macro avg       0.24      0.26      0.19       200
weighted avg       0.39      0.46      0.33       200



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [52]:
eff_base = EfficientNetB2(
    weights="imagenet",
    include_top=False,
    input_shape=(260, 260, 3),
)
eff_base.trainable = False

eff_inputs = tf.keras.Input(shape=(260, 260, 3))
y = eff_base(eff_inputs, training=False)
y = layers.GlobalAveragePooling2D()(y)
y = layers.Dense(256, activation="relu")(y)
y = layers.Dropout(0.5)(y)
eff_outputs = layers.Dense(4, activation="softmax")(y)

eff_model = tf.keras.Model(eff_inputs, eff_outputs)

eff_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

eff_model.summary()

31790344/31790344 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_11 (InputLayer)     │ (None, 260, 260, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb2 (Functional)     │ (None, 9, 9, 1408)     │     7,768,569 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_5      │ (None, 1408)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 256)            │       360,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 4)              │         1,028 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,130,301 (31.01 MB)

 Trainable params: 361,732 (1.38 MB)

 Non-trainable params: 7,768,569 (29.63 MB)

In [53]:
EFF_EPOCHS = 1
eff_history = eff_model.fit(
    eff_train_dataset,
    validation_data=eff_val_dataset,
    epochs=EFF_EPOCHS,
)

25/25 ━━━━━━━━━━━━━━━━━━━━ 112s 4s/step - accuracy: 0.3696 - loss: 1.3083 - val_accuracy: 0.5150 - val_loss: 1.0403


In [55]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_true = []
y_pred = []

for images, labels in eff_test_dataset:
    preds = eff_model.predict(images)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred))

1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
[[ 0  0 30  3]
 [ 0  0  2  0]
 [ 0  0 86  4]
 [ 0  0 67  8]]
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        33
           1       0.00      0.00      0.00         2
           2       0.46      0.96      0.63        90
           3       0.53      0.11      0.18        75

    accuracy                           0.47       200
   macro avg       0.25      0.27      0.20       200
weighted avg       0.41      0.47      0.35       200



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [56]:
vgg_probs = vgg_model.predict(vgg_test_dataset)
eff_probs = eff_model.predict(eff_test_dataset)

ensemble_probs = (vgg_probs + eff_probs) / 2.0
ensemble_preds = np.argmax(ensemble_probs, axis=1)

y_true = np.concatenate([y.numpy() for _, y in vgg_test_dataset], axis=0)

vgg_preds = np.argmax(vgg_probs, axis=1)
eff_preds = np.argmax(eff_probs, axis=1)

print("VGG16 test accuracy: ", accuracy_score(y_true, vgg_preds))
print("EfficientNetB2 test acc: ", accuracy_score(y_true, eff_preds))
print("Ensemble test accuracy: ", accuracy_score(y_true, ensemble_preds))

print("\nEnsemble classification report:")
print(classification_report(y_true, ensemble_preds, digits=4))

print("\nEnsemble confusion matrix:")
print(confusion_matrix(y_true, ensemble_preds))

7/7 ━━━━━━━━━━━━━━━━━━━━ 74s 10s/step
7/7 ━━━━━━━━━━━━━━━━━━━━ 21s 3s/step
VGG16 test accuracy:  0.455
EfficientNetB2 test acc:  0.47
Ensemble test accuracy:  0.46

Ensemble classification report:
              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000        33
           1     0.0000    0.0000    0.0000         2
           2     0.4560    0.9778    0.6219        90
           3     0.5714    0.0533    0.0976        75

    accuracy                         0.4600       200
   macro avg     0.2568    0.2578    0.1799       200
weighted avg     0.4195    0.4600    0.3164       200


Ensemble confusion matrix:
[[ 0  0 32  1]
 [ 0  0  2  0]
 [ 0  0 88  2]
 [ 0  0 71  4]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
